# Assignment 2C ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â Part A: Four Base Prompt Patterns
Implements Zero-shot, Few-shot, RAG, Self-Critique, and the ten-query benchmark. Each code cell starts with a comment describing its purpose. The reference dataset uses instruction/response fields and is streamed for the first ten questions.


## Locate the project and mount Google Drive

The setup cell mounts Drive in Colab and prefers the project copy that contains your CPT checkpoint and adapters.

In [ ]:
# This cell mounts Drive in Colab and locates the project root whether Jupyter starts in the repo, a child folder, or a Colab clone.
from pathlib import Path
import os, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
drive_project = Path("/content/drive/MyDrive/LLM_Project_1_2")
if (drive_project / "models" / "cpt_model" / "config.json").exists():
    PROJECT = drive_project
else:
    PROJECT = next(
        (parent for parent in [Path.cwd(), *Path.cwd().parents]
         if (parent / "data").exists() and (parent / "notebooks").exists()),
        None
    )
    if PROJECT is None:
        clone_path = Path("/content/banking-compliance-llm-assignment-1a")
        if not (clone_path / "src").exists():
            subprocess.run(["git", "clone", "https://github.com/tusharchouhan/banking-compliance-llm-assignment-1a.git", str(clone_path)], check=True)
        PROJECT = clone_path
os.chdir(PROJECT)
print("Project:", Path.cwd())
print("CPT checkpoint found:", (PROJECT / "models" / "cpt_model" / "config.json").exists())
print("Adapter A found:", (PROJECT / "models" / "adapter_A" / "adapter_config.json").exists())

In [ ]:
!ls

In [ ]:
!python -m pip install -r requirements.txt
!pip install faiss-cpu

In [ ]:
# This cell installs the model inference, PEFT, embedding, and FAISS retrieval dependencies.%pip install -q transformers peft accelerate sentence-transformers faiss-cpu truststore

In [ ]:
# This cell loads the Assignment 1 model or a configurable Hugging Face fallback, reads the first ten benchmark examples, and builds a FAISS index over cleaned domain text.
from pathlib import Path
import os, json, time, re, warnings
import truststore
truststore.inject_into_ssl()
import torch, numpy as np, pandas as pd, faiss
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = Path("/content/LLM_Project_1_2")
DATA = ROOT / "data"
MODEL_ID = os.getenv("MODEL_ID", "HuggingFaceTB/SmolLM2-360M")
BASE_PATH = os.getenv("BASE_MODEL_DIR", str(ROOT / "models" / "cpt_model"))
ADAPTER_PATH = os.getenv("ADAPTER_DIR", str(ROOT / "models" / "adapter_A"))
if not Path(BASE_PATH, "config.json").exists():
    BASE_PATH = MODEL_ID
tokenizer = AutoTokenizer.from_pretrained(BASE_PATH)
model = AutoModelForCausalLM.from_pretrained(
    BASE_PATH,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)
adapter_loaded = Path(ADAPTER_PATH, "adapter_config.json").exists()
if adapter_loaded:
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
else:
    warnings.warn("Assignment 1 adapter A not found; using the CPT base model only. Set ADAPTER_DIR to the adapter_A folder for required final results.")
model.eval()

def generate(prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return answer, int(output.shape[-1]), time.perf_counter() - started

dataset_path = DATA / "instruction_dataset" / "instruction_dataset.jsonl"
if not dataset_path.exists():
    raise FileNotFoundError(f"Expected benchmark file: {dataset_path}")
benchmark = []
with dataset_path.open(encoding="utf-8") as stream:
    for line in stream:
        if line.strip():
            row = json.loads(line)
            benchmark.append({
                "query": row.get("query", row.get("instruction", row.get("question", ""))),
                "answer": row.get("answer", row.get("response", ""))
            })
        if len(benchmark) == 10:
            break
if len(benchmark) < 10:
    raise ValueError("The benchmark file must contain at least ten question-answer records.")

def read_chunks(folder=DATA / "cleaned_txt", chunk_words=220, overlap=40):
    chunks = []
    step = chunk_words - overlap
    for file in folder.rglob("*.txt"):
        words = file.read_text(encoding="utf-8", errors="ignore").split()
        for start in range(0, len(words), step):
            text = " ".join(words[start:start + chunk_words])
            if text:
                chunks.append({"source": str(file.relative_to(folder)), "text": text})
    if not chunks:
        raise FileNotFoundError(f"No cleaned .txt corpus found under {folder}")
    return chunks

chunks = read_chunks()
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vectors = embedder.encode([item["text"] for item in chunks], normalize_embeddings=True, show_progress_bar=True).astype("float32")
faiss_index = faiss.IndexFlatIP(vectors.shape[1])
faiss_index.add(vectors)

def retrieve(query, k=2):
    query_vector = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = faiss_index.search(query_vector, min(k, len(chunks)))
    return [{"score": float(score), **chunks[idx]} for score, idx in zip(scores[0], ids[0]) if idx >= 0]

print(f"Questions: {len(benchmark)} | text chunks: {len(chunks)} | GPU: {torch.cuda.is_available()} | adapter loaded: {adapter_loaded}")



In [ ]:
# This cell implements the four required base patterns with comparable answer, token, and latency outputs. RAG also returns its retrieved chunks for inspection.
def zero_shot(query):
    return generate(query)

def few_shot(query, examples=None, k=3):
    examples = (examples or benchmark)[:k]
    demonstrations = "\n\n".join(
        f"Question: {item['query']}\nAnswer: {item['answer']}" for item in examples
    )
    prompt = f"Use these worked examples as guidance.\n{demonstrations}\n\nQuestion: {query}\nAnswer:"
    return generate(prompt)

def rag(query):
    hits = retrieve(query, k=2)
    context = "\n\n".join(f"[{item['source']}] {item['text']}" for item in hits)
    prompt = f"Answer using only the evidence below. If it is insufficient, say so.\nEvidence:\n{context}\n\nQuestion: {query}\nAnswer:"
    result = generate(prompt)
    return result, hits

def self_critique(query):
    draft, tokens1, latency1 = generate(f"Answer the question accurately: {query}")
    critique, tokens2, latency2 = generate(
        f"Check the draft for errors, missing points, and unsupported claims.\nQuestion: {query}\nDraft: {draft}\nCritique:"
    )
    revised, tokens3, latency3 = generate(
        f"Revise the draft using the critique. Do not add unsupported claims.\nQuestion: {query}\nDraft: {draft}\nCritique: {critique}\nRevised answer:"
    )
    return revised, tokens1 + tokens2 + tokens3, latency1 + latency2 + latency3, draft, critique

def accuracy_score(answer, reference):
    # This overlap estimate is an aid only; manually review and assign the official 0ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦ÃƒÂ¢Ã¢â€šÂ¬Ã…â€œ2 rubric.
    answer_words = set(re.findall(r"\w+", answer.lower()))
    reference_words = set(re.findall(r"\w+", reference.lower()))
    overlap = len(answer_words & reference_words) / max(1, len(reference_words))
    return 2 if overlap >= 0.65 else (1 if overlap >= 0.30 else 0)



In [ ]:
# This cell displays the retrieved top-two chunks for five queries and the draft/critique/revision for two questions, as requested by the marking checklist.
for item in benchmark[:5]:
    (rag_answer, rag_tokens, rag_latency), retrieved = rag(item["query"])
    print("QUERY:", item["query"])
    print("TOP TWO RETRIEVED CHUNKS:")
    for chunk in retrieved:
        print("SOURCE:", chunk["source"], "SCORE:", round(chunk["score"], 4))
        print(chunk["text"][:900], "\\n")
for item in benchmark[:2]:
    revised, tokens, latency, draft, critique = self_critique(item["query"])
    print("QUERY:", item["query"])
    print("DRAFT:", draft, "\\nCRITIQUE:", critique, "\\nREVISED:", revised)

In [ ]:
# This cell runs all four methods against the same ten benchmark questions and produces the required accuracy, token, and latency table.
records = []
for item in benchmark:
    query, reference = item["query"], item["answer"]
    method_outputs = {
        "Zero-shot": zero_shot(query),
        "Few-shot": few_shot(query),
        "RAG": rag(query)[0],
        "Self-Critique": self_critique(query)[:3]
    }
    for method, (answer, tokens, latency) in method_outputs.items():
        records.append({
            "query": query, "pattern": method, "answer": answer,
            "tokens": tokens, "latency_sec": latency,
            "preliminary_accuracy_0_2": accuracy_score(answer, reference),
            "human_accuracy_0_2": None
        })
part_a_results = pd.DataFrame(records)
display(part_a_results)
part_a_summary = part_a_results.groupby("pattern").agg(
    avg_preliminary_accuracy=("preliminary_accuracy_0_2", "mean"),
    total_tokens=("tokens", "sum"),
    avg_latency_sec=("latency_sec", "mean")
).reset_index()
display(part_a_summary)



## Required 75-word analysis

Pending final run: rerun this notebook with adapter_loaded=True, replace preliminary scores with reviewed 0ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã…â€œ2 scores, and then write the 75-word comparison from the resulting accuracy, token, and latency table.

### Save and Push Changes Back to Git
Configure your Git credentials below, stage all changes, commit, and push back to your remote repository.

## Save and submit

This notebook does not commit or push to GitHub automatically. After reviewing outputs, save the notebook and use your normal Git workflow to submit it.